# Magnetic Field Evolution in Satellite Galaxies
## PHY 225-001 — Computational Physics Project

**Goal:** Track magnetic field strength as a function of redshift for satellite galaxies in TNG100, using merger tree data to identify the moment of central-to-satellite transition (infall time).

---
### Notebook Structure
1. Setup & Imports
2. Build Snapshot → Redshift / Lookback-Time Table
3. Query Satellite Galaxies at z=0
4. Walk the Merger Tree (main progenitor branch)
5. Identify Infall Time (central → satellite transition)
6. Plot B-field vs Redshift (single galaxy)
7. Generalize to Multiple Galaxies
8. Averaged B-field vs Time Since Infall
9. (Reach Goal) Fit Exponential Growth Model

---
## 1. Setup & Imports

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit

# Make sure iapi.py is in the same directory as this notebook
import iapi

# ── TNG100 cosmological parameters (Planck 2015) ──────────────────────────────
cosmo    = FlatLambdaCDM(H0=67.74, Om0=0.3089)
SIM_NAME = 'TNG100-1'
h_cosmo  = 0.6774   # dimensionless Hubble parameter

# Make sure output folders exist
os.makedirs('trees', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

# ── Quick connectivity test ───────────────────────────────────────────────────
try:
    info = iapi.get(iapi.baseUrl + SIM_NAME + '/')
    print(f'Connected to {SIM_NAME}')
    print(f'  Number of snapshots : {info["num_snapshots"]}')
    print(f'  Box size            : {info["boxsize"]:.1f} ckpc/h')
    #print(f'  Available keys: {list(info.keys())}')  # see what's actually returned
except Exception as e:
    print(f'Connection failed — check iapi.py headers: {e}')

Connected to TNG100-1
  Number of snapshots : 100
  Box size            : 75000.0 ckpc/h


---
## 2. Build the Snapshot → Redshift / Lookback-Time Table

TNG100 has **100 snapshots** (snap 0 = high-z, snap 99 = z=0).  
We download the metadata once and store it for all later conversions.

In [4]:
def build_snapshot_table(sim_name=SIM_NAME):
    """
    Return a dict mapping snapshot number -> {redshift, scale_factor, lookback_time_Gyr}.
    Uses iapi.get() for the HTTP request.
    """
    snaps_info = iapi.get(iapi.baseUrl + sim_name + '/snapshots/')
    table = {}
    for s in snaps_info:
        z  = s['redshift']
        lb = cosmo.lookback_time(z).to(u.Gyr).value
        table[s['number']] = {
            'redshift'      : z,
            'scale_factor'  : 1.0 / (1.0 + z),
            'lookback_time' : lb,    # Gyr
        }
    return table

SNAP_TABLE = build_snapshot_table()

# Preview a few rows
print(f"{'Snap':>5}  {'z':>8}  {'a':>8}  {'Lookback (Gyr)':>15}")
print('-' * 45)
for sn in [0, 10, 33, 50, 67, 78, 91, 99]:
    row = SNAP_TABLE[sn]
    print(f"{sn:>5}  {row['redshift']:>8.3f}  {row['scale_factor']:>8.4f}  {row['lookback_time']:>15.3f}")

 Snap         z         a   Lookback (Gyr)
---------------------------------------------
    0    20.046    0.0475           13.623
   10     7.236    0.1214           13.071
   33     2.002    0.3331           10.518
   50     0.997    0.5007            7.925
   67     0.503    0.6653            5.216
   78     0.298    0.7706            3.504
   91     0.099    0.9096            1.336
   99     0.000    1.0000            0.000


---
## 3. Query Satellite Galaxies at z = 0 (Snapshot 99)

We select **satellite** subhalos from the z=0 snapshot.  
A subhalo is a satellite if `primary_flag=0` — meaning it is NOT the most massive subhalo in its FoF group.

In [7]:
print(iapi.get(satellites_z0[0]['url']).keys())

dict_keys(['snap', 'id', 'bhmdot', 'cm_x', 'cm_y', 'cm_z', 'gasmetallicity', 'gasmetallicityhalfrad', 'gasmetallicitymaxrad', 'gasmetallicitysfr', 'gasmetallicitysfrweighted', 'pos_x', 'pos_y', 'pos_z', 'halfmassrad', 'halfmassrad_gas', 'halfmassrad_dm', 'halfmassrad_stars', 'halfmassrad_bhs', 'len', 'len_gas', 'len_dm', 'len_stars', 'len_bhs', 'mass', 'mass_gas', 'mass_dm', 'mass_stars', 'mass_bhs', 'massinhalfrad', 'massinhalfrad_gas', 'massinhalfrad_dm', 'massinhalfrad_stars', 'massinhalfrad_bhs', 'massinmaxrad', 'massinmaxrad_gas', 'massinmaxrad_dm', 'massinmaxrad_stars', 'massinmaxrad_bhs', 'massinrad', 'massinrad_gas', 'massinrad_dm', 'massinrad_stars', 'massinrad_bhs', 'sfr', 'sfrinhalfrad', 'sfrinmaxrad', 'sfrinrad', 'spin_x', 'spin_y', 'spin_z', 'starmetallicity', 'starmetallicityhalfrad', 'starmetallicitymaxrad', 'stellarphotometrics_u', 'stellarphotometrics_b', 'stellarphotometrics_v', 'stellarphotometrics_k', 'stellarphotometrics_g', 'stellarphotometrics_r', 'stellarphotome

In [6]:
def query_satellites(snap=99, stellar_mass_min_log=9.5, stellar_mass_max_log=11.5, limit=20):
    """
    Return a list of satellite subhalo dicts from a given snapshot.
    Uses mass_log_msun for filtering since that's what the API returns by default.
    """
    url    = iapi.baseUrl + SIM_NAME + f'/snapshots/{snap}/subhalos/'
    params = {
        'primary_flag'          : 0,
        'mass_log_msun__gte'    : stellar_mass_min_log,
        'mass_log_msun__lte'    : stellar_mass_max_log,
        'limit'                 : limit,
    }
    result   = iapi.get(url, params=params)
    galaxies = result.get('results', [])
    print(f"Found {result['count']} satellites matching criteria; returning {len(galaxies)}.")
    return galaxies


def get_subhalo_details(subhalo_url):
    """
    Fetch full field data for a single subhalo using its URL.
    Returns B field and stellar mass from the detailed subhalo endpoint.
    """
    data = iapi.get(subhalo_url)
    return {
        'id'                  : data['id'],
        'SubhaloMagneticField': data['SubhaloMagneticField'],
        'M_star'              : 10**data['mass_log_msun'],   # Msun
        'url'                 : subhalo_url,
    }


# Re-run the query
satellites_z0 = query_satellites(snap=99, stellar_mass_min_log=9.5, limit=10)

# Fetch full details for each (one API call per galaxy)
print('\nFetching full subhalo details...')
satellites_detailed = []
for g in satellites_z0:
    details = get_subhalo_details(g['url'])
    satellites_detailed.append(details)

# Display
print(f"\n{'SubhaloID':>10}  {'log M*':>8}  {'B_comoving (G)':>15}")
print('-' * 38)
for g in satellites_detailed:
    print(f"{g['id']:>10d}  {np.log10(g['M_star']):>8.2f}  {g['SubhaloMagneticField']:>15.4e}")

Found 940505 satellites matching criteria; returning 10.

Fetching full subhalo details...


KeyError: 'SubhaloMagneticField'

In [5]:
def query_satellites(snap=99, stellar_mass_min_log=9.5, stellar_mass_max_log=11.5, limit=20):
    """
    Return a list of satellite subhalo dicts from a given snapshot.

    Parameters
    ----------
    snap                 : TNG snapshot number (99 = z=0)
    stellar_mass_min_log : log10(M*/Msun) lower bound
    stellar_mass_max_log : log10(M*/Msun) upper bound
    limit                : max number of subhalos to return (start small!)
    """
    mass_unit = 1e10 / h_cosmo   # Msun per TNG code unit
    m_min = 10**stellar_mass_min_log / mass_unit
    m_max = 10**stellar_mass_max_log / mass_unit

    url    = iapi.baseUrl + SIM_NAME + f'/snapshots/{snap}/subhalos/'
    params = {
        'primary_flag'         : 0,
        'SubhaloMassType__gte' : f'4,{m_min:.6f}',
        'SubhaloMassType__lte' : f'4,{m_max:.6f}',
        'fields'               : 'id,SubhaloMassType,SubhaloMagneticField,SubhaloGrNr',
        'limit'                : limit,
    }
    result   = iapi.get(url, params=params)
    galaxies = result.get('results', [])
    print(f"Found {result['count']} satellites matching criteria; returning {len(galaxies)}.")
    return galaxies


# Start with a small sample while you explore
satellites_z0 = query_satellites(snap=99, stellar_mass_min_log=9.5, limit=10)
# Check what fields actually came back
print(satellites_z0[0].keys())
print(satellites_z0[0])

print()
print(f"{'SubhaloID':>10}  {'log M*':>8}  {'B_comoving (G)':>15}")
print('-' * 38)
for g in satellites_z0:
    M_star = g['SubhaloMassType'][4] * 1e10 / h_cosmo
    B      = g['SubhaloMagneticField']
    print(f"{g['id']:>10d}  {np.log10(M_star):>8.2f}  {B:>15.4e}")

Found 940505 satellites matching criteria; returning 10.
dict_keys(['id', 'sfr', 'mass_log_msun', 'url'])
{'id': 1, 'sfr': 0.413285, 'mass_log_msun': 13.733420710710462, 'url': 'http://www.tng-project.org/api/TNG100-1/snapshots/99/subhalos/1/'}

 SubhaloID    log M*   B_comoving (G)
--------------------------------------


KeyError: 'SubhaloMassType'

---
## 4. Walk the Merger Tree — Main Progenitor Branch

We use `iapi.gettree()` which:
- Downloads the MPB as an `.hdf5` file via the TNG API
- **Caches it** in the `trees/` folder — so re-runs don't re-hit the API

We then read the HDF5 file with `h5py` and extract the fields we need at each snapshot.

**Fields extracted:**

| Field | Description |
|---|---|
| `SnapNum` | Snapshot number |
| `SubhaloMagneticField` | Volume-weighted mean B-field (comoving Gauss) |
| `SubhaloMassType` | Mass by particle type (index 4 = stellar) |
| `GroupFirstSub` | SubhaloID of the primary subhalo of the host FoF group |
| `SubfindID` | The subhalo's own ID at that snapshot |

> **Note on units:** `SubhaloMagneticField` is in comoving Gauss.  
> To convert to physical Gauss: **B_phys = B_com / a²** where a is the scale factor.

In [ ]:
def get_mpb(subhalo_id, snap=99):
    """
    Retrieve the Main Progenitor Branch for a subhalo using iapi.gettree().
    Returns a list of dicts, one per snapshot (z=0 first, high-z last).
    """
    tree_file = iapi.gettree(snap, subhalo_id)   # downloads & caches HDF5

    records = []
    with h5py.File(tree_file, 'r') as f:
        snaps     = f['SnapNum'][:]
        B_vals    = f['SubhaloMagneticField'][:]
        masses    = f['SubhaloMassType'][:]       # shape (N, 6)
        sub_ids   = f['SubfindID'][:]
        grp_first = f['GroupFirstSub'][:]

        for i in range(len(snaps)):
            sn = int(snaps[i])
            sd = SNAP_TABLE.get(sn, {})
            records.append({
                'snap'            : sn,
                'redshift'        : sd.get('redshift', np.nan),
                'lookback_time'   : sd.get('lookback_time', np.nan),
                'scale_factor'    : sd.get('scale_factor', np.nan),
                'B_comoving'      : float(B_vals[i]),
                'M_star'          : float(masses[i][4]) * 1e10 / h_cosmo,  # Msun
                'subfind_id'      : int(sub_ids[i]),
                'group_first_sub' : int(grp_first[i]),
            })
    return records


def compute_B_physical(records):
    """
    Add B_physical (Gauss) to each record.
    B_phys = B_comoving / a^2
    """
    for r in records:
        a = r['scale_factor']
        r['B_physical'] = r['B_comoving'] / a**2 if a > 0 else np.nan
    return records


# ── Test on the first satellite ───────────────────────────────────────────────
test_id  = satellites_z0[0]['id']
mpb_data = get_mpb(test_id)
mpb_data = compute_B_physical(mpb_data)

print(f"MPB for SubhaloID {test_id}: {len(mpb_data)} snapshots")
print()
print(f"{'Snap':>5}  {'z':>7}  {'B_com (G)':>12}  {'B_phys (G)':>12}  {'log M*':>8}")
print('-' * 55)
for r in mpb_data[:8]:
    logM = np.log10(r['M_star']) if r['M_star'] > 0 else np.nan
    print(f"{r['snap']:>5}  {r['redshift']:>7.3f}  "
          f"{r['B_comoving']:>12.4e}  {r['B_physical']:>12.4e}  {logM:>8.2f}")

---
## 5. Identify Infall Time (Central → Satellite Transition)

A galaxy **becomes a satellite** when it stops being the primary subhalo of its FoF group.  
We walk the MPB from z=0 backwards and find where `subfind_id == group_first_sub` (was a central).  
The **infall snapshot** is just before that point.

In [ ]:
def find_infall_snap(mpb_records):
    """
    Identify the snapshot of central -> satellite transition.

    MPB is ordered z=0 -> high-z (index 0 is z=0).
    Returns a dict with snap, redshift, lookback_time -- or None if not found.
    """
    was_central_at = None
    for i, r in enumerate(mpb_records):
        if r['subfind_id'] == r['group_first_sub']:
            was_central_at = i   # most recent snapshot where it was a central

    if was_central_at is None or was_central_at == 0:
        return None   # always a satellite, or still central at z=0

    infall_record = mpb_records[was_central_at - 1]
    return {
        'snap'         : infall_record['snap'],
        'redshift'     : infall_record['redshift'],
        'lookback_time': infall_record['lookback_time'],
    }


infall = find_infall_snap(mpb_data)

if infall:
    print(f"Infall snapshot : {infall['snap']}")
    print(f"Infall redshift : z = {infall['redshift']:.3f}")
    print(f"Infall lookback : {infall['lookback_time']:.2f} Gyr ago")
else:
    print('Could not determine infall time for this galaxy.')

---
## 6. Plot B-field vs Redshift — Single Galaxy

- Plot B_physical vs redshift along the MPB
- Mark the infall redshift with a vertical line
- Overlay a cubic interpolation for a smooth curve

In [ ]:
def plot_B_vs_redshift_single(mpb_records, infall_info=None, subhalo_id=None, smooth=True):
    """
    Plot magnetic field strength vs redshift for a single galaxy's MPB.
    """
    valid  = [r for r in mpb_records if np.isfinite(r['B_physical']) and r['B_physical'] > 0]
    z_arr  = np.array([r['redshift']   for r in valid])
    B_arr  = np.array([r['B_physical'] for r in valid])
    order  = np.argsort(z_arr)
    z_arr, B_arr = z_arr[order], B_arr[order]

    # Cubic interpolation onto a fine grid
    z_fine   = np.linspace(z_arr.min(), z_arr.max(), 500)
    B_interp = interp1d(z_arr, B_arr, kind='cubic', fill_value='extrapolate')
    B_fine   = B_interp(z_fine)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(z_arr, B_arr, s=30, color='steelblue', zorder=3, label='TNG100 snapshots')
    ax.plot(z_arr, B_arr, color='steelblue', alpha=0.4, lw=1)

    if smooth:
        ax.plot(z_fine, B_fine, color='navy', lw=2, ls='--', label='Cubic interpolation')

    if infall_info:
        ax.axvline(infall_info['redshift'], color='tomato', lw=2,
                   label=f"Infall  z = {infall_info['redshift']:.2f}")

    ax.set_yscale('log')
    ax.set_xlabel('Redshift $z$', fontsize=13)
    ax.set_ylabel(r'$B_{\rm phys}$ [G]', fontsize=13)
    ax.set_title(f'Magnetic Field Evolution -- Subhalo {subhalo_id}', fontsize=14)
    ax.invert_xaxis()   # higher redshift (older) on the right
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/figures/B_vs_z_single.png', dpi=150)
    plt.show()


plot_B_vs_redshift_single(mpb_data, infall_info=infall, subhalo_id=test_id)

---
## 7. Generalize to Multiple Galaxies

Loop over all queried satellites, fetch each MPB via `iapi.gettree()`, and store results.  
Already-downloaded trees are loaded from the `trees/` cache — no repeat API calls.

In [ ]:
def process_all_satellites(satellite_list):
    """
    For each satellite, fetch MPB, compute physical B-field, and find infall.
    Returns a list of dicts: {subhalo_id, mpb, infall}
    """
    results = []
    for i, gal in enumerate(satellite_list):
        sid = gal['id']
        print(f'[{i+1}/{len(satellite_list)}] Processing SubhaloID {sid} ...', end=' ')
        try:
            mpb    = get_mpb(sid)
            mpb    = compute_B_physical(mpb)
            infall = find_infall_snap(mpb)
            results.append({'subhalo_id': sid, 'mpb': mpb, 'infall': infall})
            status = f"infall z={infall['redshift']:.2f}" if infall else 'no infall found'
            print(f'OK  ({status})')
        except Exception as e:
            print(f'FAILED: {e}')
    print(f'\nProcessed  : {len(results)} galaxies')
    print(f'With infall: {sum(1 for g in results if g["infall"])}')
    return results


all_galaxies = process_all_satellites(satellites_z0)

In [ ]:
def plot_B_vs_z_all(all_galaxies):
    """
    Overlay MPB B-field tracks for all galaxies, color-coded by infall redshift.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    infalls  = [g['infall']['redshift'] for g in all_galaxies if g['infall']]
    z_min, z_max = (min(infalls), max(infalls)) if infalls else (0, 2)
    cmap     = plt.cm.plasma
    norm     = plt.Normalize(vmin=z_min, vmax=z_max)

    for gal in all_galaxies:
        valid = [r for r in gal['mpb'] if np.isfinite(r['B_physical']) and r['B_physical'] > 0]
        if not valid:
            continue
        z_arr = np.array([r['redshift']   for r in valid])
        B_arr = np.array([r['B_physical'] for r in valid])
        order = np.argsort(z_arr)
        color = cmap(norm(gal['infall']['redshift'])) if gal['infall'] else 'gray'
        ax.plot(z_arr[order], B_arr[order], color=color, alpha=0.55, lw=1.5)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Infall redshift $z_{\\rm infall}$')

    ax.set_yscale('log')
    ax.set_xlabel('Redshift $z$', fontsize=13)
    ax.set_ylabel(r'$B_{\rm phys}$ [G]', fontsize=13)
    ax.set_title('Magnetic Field Evolution -- All Sampled Satellites (TNG100)', fontsize=14)
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/figures/B_vs_z_all.png', dpi=150)
    plt.show()


plot_B_vs_z_all(all_galaxies)

---
## 8. B-field vs Time Since Infall (Key Science Plot)

Re-align all tracks so **Δt = 0 at infall**, then compute the median + scatter across galaxies.

- **Positive Δt** = before infall (galaxy was still a central)
- **Negative Δt** = after infall (galaxy is now a satellite)

In [ ]:
def compute_dt_infall(mpb_records, infall_info):
    """
    Add dt_infall = lookback_time - lookback_time_infall to each record.
    Positive dt -> before infall. Negative dt -> after infall.
    """
    if infall_info is None:
        return None
    t_inf = infall_info['lookback_time']
    for r in mpb_records:
        r['dt_infall'] = r['lookback_time'] - t_inf
    return mpb_records


# Add dt_infall to all galaxies
for gal in all_galaxies:
    compute_dt_infall(gal['mpb'], gal['infall'])


def plot_B_vs_dt_infall(all_galaxies, dt_range=(-4, 6), bin_width=0.5):
    """
    Plot median B-field (+ 1-sigma band) as a function of time since infall.
    Individual galaxy tracks shown faintly in the background.
    """
    all_dt, all_B = [], []
    for gal in all_galaxies:
        if not gal['infall']:
            continue
        for r in gal['mpb']:
            if 'dt_infall' in r and np.isfinite(r['B_physical']) and r['B_physical'] > 0:
                all_dt.append(r['dt_infall'])
                all_B.append(r['B_physical'])

    all_dt = np.array(all_dt)
    all_B  = np.array(all_B)

    # Bin and compute median + 16th/84th percentile
    bins   = np.arange(dt_range[0], dt_range[1] + bin_width, bin_width)
    bin_c  = 0.5 * (bins[:-1] + bins[1:])
    median, p16, p84 = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (all_dt >= lo) & (all_dt < hi)
        if mask.sum() > 1:
            vals = all_B[mask]
            median.append(np.median(vals))
            p16.append(np.percentile(vals, 16))
            p84.append(np.percentile(vals, 84))
        else:
            median.append(np.nan); p16.append(np.nan); p84.append(np.nan)

    median = np.array(median)
    p16    = np.array(p16)
    p84    = np.array(p84)

    fig, ax = plt.subplots(figsize=(10, 5))

    # Individual tracks (faint background)
    for gal in all_galaxies:
        if not gal['infall']:
            continue
        valid = [r for r in gal['mpb']
                 if 'dt_infall' in r and np.isfinite(r['B_physical']) and r['B_physical'] > 0]
        if not valid:
            continue
        dt_g  = np.array([r['dt_infall']  for r in valid])
        B_g   = np.array([r['B_physical'] for r in valid])
        order = np.argsort(dt_g)
        ax.plot(dt_g[order], B_g[order], color='steelblue', alpha=0.15, lw=1)

    # Median + scatter band
    valid_bins = np.isfinite(median)
    ax.fill_between(bin_c[valid_bins], p16[valid_bins], p84[valid_bins],
                    alpha=0.35, color='tomato', label='16th-84th percentile')
    ax.plot(bin_c[valid_bins], median[valid_bins],
            color='tomato', lw=2.5, label='Median')

    ax.axvline(0, color='black', lw=2, ls='--', label='Infall ($\\Delta t = 0$)')

    ax.set_yscale('log')
    ax.set_xlabel(r'$\Delta t = t_{\rm lookback} - t_{\rm infall}$ [Gyr]', fontsize=13)
    ax.set_ylabel(r'$B_{\rm phys}$ [G]', fontsize=13)
    ax.set_title('Magnetic Field vs Time Since Infall -- TNG100 Satellites', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/figures/B_vs_dt_infall.png', dpi=150)
    plt.show()


plot_B_vs_dt_infall(all_galaxies)

---
## 9. (Reach Goal) Fit Exponential Growth to Post-Infall B-field

Fit **B(Δt) = B₀ · exp(−Δt / τ)** to the post-infall (Δt < 0) median,  
where τ is the **amplification timescale** in Gyr.

In [ ]:
def exponential_growth(dt, B0, tau):
    """B0 * exp(-dt / tau) -- dt is negative after infall, so B grows toward z=0."""
    return B0 * np.exp(-dt / tau)


def fit_amplification_timescale(all_galaxies, bin_width=0.5):
    """
    Fit an exponential to the post-infall (dt < 0) binned median B-field.
    """
    all_dt, all_B = [], []
    for gal in all_galaxies:
        if not gal['infall']:
            continue
        for r in gal['mpb']:
            if ('dt_infall' in r and r['dt_infall'] < 0
                    and np.isfinite(r['B_physical']) and r['B_physical'] > 0):
                all_dt.append(r['dt_infall'])
                all_B.append(r['B_physical'])

    all_dt = np.array(all_dt)
    all_B  = np.array(all_B)

    dt_min = all_dt.min() if len(all_dt) else -4
    bins   = np.arange(dt_min, 0 + bin_width, bin_width)
    bin_c  = 0.5 * (bins[:-1] + bins[1:])
    median = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (all_dt >= lo) & (all_dt < hi)
        median.append(np.median(all_B[mask]) if mask.sum() > 1 else np.nan)

    median = np.array(median)
    valid  = np.isfinite(median)
    x, y   = bin_c[valid], median[valid]

    if len(x) < 3:
        print('Not enough bins to fit -- try a larger galaxy sample.')
        return

    try:
        popt, pcov = curve_fit(exponential_growth, x, y, p0=[y[-1], 2.0], maxfev=5000)
        B0_fit, tau_fit = popt
        perr = np.sqrt(np.diag(pcov))
        print(f'Best-fit amplification timescale : tau = {tau_fit:.2f} +/- {perr[1]:.2f} Gyr')
        print(f'B at infall (B0)                 : B0  = {B0_fit:.3e} G')

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(x, y, color='tomato', zorder=3, label='Binned median (post-infall)')
        t_fine = np.linspace(x.min(), 0, 300)
        ax.plot(t_fine, exponential_growth(t_fine, *popt), 'k--', lw=2,
                label=rf'Fit: $\tau = {tau_fit:.2f}$ Gyr')
        ax.axvline(0, color='gray', ls=':', label='Infall')
        ax.set_yscale('log')
        ax.set_xlabel(r'$\Delta t$ [Gyr]  (negative = post-infall)', fontsize=12)
        ax.set_ylabel(r'$B_{\rm phys}$ [G]', fontsize=12)
        ax.set_title('Post-Infall Magnetic Field Amplification Fit', fontsize=13)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('results/figures/B_amplification_fit.png', dpi=150)
        plt.show()

    except RuntimeError as e:
        print(f'Fit failed: {e}')


fit_amplification_timescale(all_galaxies)

---
## Notes & Next Steps

### Caching
`iapi.gettree()` already saves HDF5 files to `trees/` automatically. To avoid re-querying the satellite list too, you can pickle `all_galaxies`:

```python
import pickle
# Save
with open('data/processed/all_galaxies.pkl', 'wb') as f:
    pickle.dump(all_galaxies, f)
# Reload later without any API calls
with open('data/processed/all_galaxies.pkl', 'rb') as f:
    all_galaxies = pickle.load(f)
```

### Scaling up
Change `limit=10` in `query_satellites()` to 100-500 once you're happy with the pipeline. Each `gettree()` call takes ~1-2 seconds but is cached after the first run.

### Unit check
Verify the `SubhaloMagneticField` definition for TNG100-1 in the [TNG field specifications](https://www.tng-project.org/data/docs/specifications/#sec2a) to confirm the B_phys = B_com / a^2 conversion is correct.

### Stratified analysis (reach goal)
Split by stellar mass bins and compare tau across mass bins to test whether more massive satellites amplify their fields faster.